# Unit 14 — Two Pointers & Sliding Window

A school store has a sorted list of prices and a fixed budget.
Can you find two prices that spend the budget exactly without testing every pair?
Later, a training log asks for the longest consecutive stretch whose total stays under a limit.
Both problems become linear when two indices move forward with a clear rule.

## Lesson 1 — Converging Two Pointers

Start with a list sorted from smallest to largest.
Put `lo` at the first value and `hi` at the last value.
If `values[lo] + values[hi]` is too small, move `lo` right because keeping the smallest value cannot make a larger sum.
If the sum is too large, move `hi` left because keeping the largest value cannot make a smaller sum.
If the sum equals the target, the pair has been found.
The loop continues only while `lo < hi`, so one value is never used twice.

## Hand Trace a Pair Search

For prices `[1, 2, 4, 8, 16, 32]` and target `12`, begin with `1 + 32`.
That is too large, so move `hi` left twice until the sum is `1 + 8`.
That is too small, so move `lo` right twice.
The final possible comparison is `4 + 8`, which finds the target.
Every move permanently rules out one endpoint, so at most `N - 1` comparisons are needed.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    i = 0
    while i < n:
        values.append(int(tokens[i + 2]))
        i = i + 1

    lo = 0
    hi = n - 1
    while lo < hi:
        pair_sum = values[lo] + values[hi]
        if pair_sum == target:
            return "YES"
        if pair_sum < target:
            lo = lo + 1
        else:
            hi = hi - 1
    return "NO"

assert solve("6 12 1 2 4 8 16 32") == "YES"
assert solve("5 20 2 5 9 12 30") == "NO"

## Closest Pairs and One-Way Decisions

The same inward movement can find the pair whose sum is closest to a target.
Before moving a pointer, compare the current distance `abs(pair_sum - target)` with the best distance seen so far.
Then move `lo` right when the sum is too small, or `hi` left when it is too large.
A useful test places the best pair near the middle, after several moves from both ends.
Other problems use the same one-way decision: for example, the lightest and heaviest students either share a boat or the heaviest student must ride alone.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    i = 0
    while i < n:
        values.append(int(tokens[i + 2]))
        i = i + 1

    lo = 0
    hi = n - 1
    best_distance = abs(values[lo] + values[hi] - target)
    while lo < hi:
        pair_sum = values[lo] + values[hi]
        distance = abs(pair_sum - target)
        if distance < best_distance:
            best_distance = distance
        if pair_sum < target:
            lo = lo + 1
        else:
            hi = hi - 1
    return str(best_distance)

assert solve("6 20 1 4 7 12 18 30") == "1"
assert solve("2 10 3 7") == "0"

## Why Converging Pointers Are O(N)

A nested-loop search may test nearly every pair, which is O(N squared).
With converging pointers, each comparison moves either `lo` right or `hi` left.
Neither pointer reverses direction, and together they move at most `N - 1` times.
The scan is therefore O(N) after the list is sorted.
If sorting is required first, sorting costs O(N log N) and dominates the O(N) scan.

## Lesson 2 — Sliding Windows

A sliding window is a contiguous run from index `left` through index `right`.
Grow the window by moving `right` one step and adding the new value to `window_sum`.
When the total is too large, shrink from the left: subtract `values[left]`, then move `left` right.
Update the running sum incrementally on every move.
Do not recompute the whole window and do not replace the pointer movement with prefix sums.

## Non-Negative Values Make Shrinking Safe

For a window-sum problem, every value must be non-negative.
Adding a non-negative value cannot lower the sum, and removing one cannot raise it.
That monotonic behavior proves that an over-limit window must shrink and that discarded left endpoints never need to return.
With negative values, adding a later value could lower the total, so this grow-and-shrink rule can discard the correct answer.
Always check the constraints before using a sum-based sliding window.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    limit = int(tokens[1])
    values = []
    i = 0
    while i < n:
        values.append(int(tokens[i + 2]))
        i = i + 1

    left = 0
    right = 0
    window_sum = 0
    best_length = 0
    while right < n:
        window_sum = window_sum + values[right]
        while window_sum > limit and left <= right:
            window_sum = window_sum - values[left]
            left = left + 1
        current_length = right - left + 1
        if current_length > best_length:
            best_length = current_length
        right = right + 1
    return str(best_length)

assert solve("5 3 4 3 1 1 1") == "3"
assert solve("4 0 2 5 1 9") == "0"

## Shortest Window Reverses the Goal

To find the shortest non-empty window with sum at least a target, still grow by moving `right`.
Whenever the sum is large enough, record the current length and keep shrinking from `left` while the condition remains true.
This finds the shortest valid window ending at the current `right`.
Use a sentinel such as `N + 1` for the best length, then return `0` if no valid window was ever found.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    i = 0
    while i < n:
        values.append(int(tokens[i + 2]))
        i = i + 1

    left = 0
    right = 0
    window_sum = 0
    best_length = n + 1
    while right < n:
        window_sum = window_sum + values[right]
        while window_sum >= target and left <= right:
            current_length = right - left + 1
            if current_length < best_length:
                best_length = current_length
            window_sum = window_sum - values[left]
            left = left + 1
        right = right + 1
    if best_length == n + 1:
        return "0"
    return str(best_length)

assert solve("5 9 5 1 1 1 7") == "3"
assert solve("4 20 2 3 4 5") == "0"

## Why a Sliding Window Is O(N)

A nested loop can choose every left endpoint and scan every right endpoint, taking O(N squared) time.
In a sliding window, `right` crosses each position once and `left` also crosses each position at most once.
An inner shrinking loop does not make the algorithm quadratic because all of its pointer moves total at most `N` over the entire run.
The complete scan is O(N) time and uses O(1) extra space beyond the input list.

## Two-Pointer Checklist

For converging pointers, confirm the data is sorted and explain why each comparison safely discards one endpoint.
For a sliding window, define exactly what lies between `left` and `right`, update its sum or count when either pointer moves, and test a case that must shrink.
For sum conditions, confirm every value is non-negative.
Test a pair found at the last inward step, a valid answer ending at the final value, and a window that shrinks until it becomes empty.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))